In [1]:
import numpy as np
import matplotlib.pyplot as plt
from FastBEMT.JobParameters import LowFidelityParameters
from FastBEMT.Propeller import Propeller
from FastBEMT.DataLoader import load_propeller_dict
from PropGen.BladeDictGenerator import generate_blade_dict
from PropGen.GAN import GAN
import pickle
import scienceplots
plt.style.use(['science','no-latex'])

# blade_dict = load_propeller_dict('run_001/Pareto1')

params = LowFidelityParameters(
    rpm=7000,
    a_inf=343,
    rho=1.225,
    mu=1.81e-5,
    n_blades=2,
    p_ref=2e-5,
    revolutions=5,
    num_obs_times_per_rev=50,
    device='cuda',
)

In [2]:
gan = GAN(latent_dim=4, noise_dim=0)
# gan.load_checkpoint('N:/Teams/Groups/ThermalSim/Projects/Public/AirfoilGAN Models/WGAN_LCR1/latent4_noise0/checkpoint_50000.pth')
gan.load_checkpoint('C:/Users/Davide Pusino/Desktop/PropGen/Datasets/checkpoint_50000.pth')

Successfully loaded checkpoint from step 50000


In [3]:
with open('../Datasets/Propellers/run_001.pkl', 'rb') as f:
    data = pickle.load(f)

c:\Users\Davide Pusino\Desktop\FastBEMT\.FastBEMT\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [ ]:
x_original = data.X[0]

c_block = x_original[0:4]   # chord (4 numbers)
d_block = x_original[4:7]   # twist (3 numbers)
a_block = x_original[7:11]  # root (4 numbers)
b_block = x_original[11:15] # tip (4 numbers)

x_reordered = np.concatenate([a_block, b_block, c_block, d_block])

print(f"Original order (c,d,a,b): {x_original}")
print(f"Reordered (a,b,c,d): {x_reordered}")

In [ ]:
blade_dict = generate_blade_dict(
    x=x_reordered,
    mode='parametric',
    gan=gan,
    r_hub = 0.02794,
    r_tip = 0.127
)

Original order (c,d,a,b): [2.85803831e-02 2.02671958e+01 6.77574801e-03 4.98856587e-03
 3.86742381e+01 4.89174026e-01 5.62719104e+00 8.71002853e-02
 2.51682845e-01 9.65440864e-01 4.94611521e-01 6.72404755e-01
 4.03562175e-01 5.64895127e-01 1.56875937e-01]
Reordered (a,b,c,d): [8.71002853e-02 2.51682845e-01 9.65440864e-01 4.94611521e-01
 6.72404755e-01 4.03562175e-01 5.64895127e-01 1.56875937e-01
 2.85803831e-02 2.02671958e+01 6.77574801e-03 4.98856587e-03
 3.86742381e+01 4.89174026e-01 5.62719104e+00]


In [11]:
blade_dict['chord']

[0.28575242248364646,
 0.35446269196360536,
 0.40367466120027784,
 0.4428066511420051,
 0.47503222850858295,
 0.5019906625083581,
 0.5246885230192919,
 0.5438080420765419,
 0.5598425502144639,
 0.5731655583365697,
 0.5840697571433952,
 0.5927907243048042,
 0.5995221744766515,
 0.6044262183505404,
 0.6076405198538088,
 0.6092834423046124,
 0.6094578439577136,
 0.6082539389987887,
 0.6057514950906475,
 0.6020215493332631,
 0.5971277677622844,
 0.5911275364115867,
 0.5840728470994682,
 0.5760110240612337,
 0.5669853256464081,
 0.557035446832133,
 0.5461979421854911,
 0.5345065844206617,
 0.5219926703626374,
 0.5086852836212321]

In [5]:
propeller = Propeller(
    geometry=blade_dict,
    params=params,
)
v_inf = 0
propeller.run_bemt(v_inf=v_inf)

In [7]:
propeller.compute_total_forces()

(np.float64(0.00015675022540682882),
 np.float64(0.032079690589360964),
 np.float64(2.2586224447992158e-06),
 np.float64(0.01143435806977699))

In [8]:
from FastBEMT.Stress import BladeStressCalculator

stress_calc = BladeStressCalculator(propeller=propeller)
sigma_c, sigma_b = stress_calc.blade_stress_report(material_rho=2700, show = False)

In [9]:
import matplotlib.pyplot as plt
from matplotlib.colors import Normalize
import matplotlib as mpl
cmap = mpl.colormaps['viridis']

geom = propeller.geometry
r = np.asarray(geom['r'])
chord = np.asarray(geom['chord'])
twist = np.radians(np.asarray(geom['twist']))
airfoils = geom['airfoil']

sigma_c = np.asarray(sigma_c)
sigma_b = np.asarray(sigma_b)

if sigma_b.ndim == 1:
    sigma_total = sigma_c + sigma_b
    sigma_total = sigma_total[:, np.newaxis]
else:
    sigma_total = sigma_b + sigma_c[:, np.newaxis]

n_sections = len(r)
n_points = airfoils[0].shape[0]

X = np.zeros((n_sections, n_points))
Y = np.zeros((n_sections, n_points))
Z = np.zeros((n_sections, n_points))
S = np.zeros((n_sections, n_points))

for i in range(n_sections):
    coords = np.asarray(airfoils[i])
    x_local = coords[:, 0] * chord[i]
    z_local = coords[:, 1] * chord[i]

    if hasattr(propeller, 'com_shift_forward') and hasattr(propeller, 'com_shift_up'):
        x_local = x_local + propeller.com_shift_forward[i] * chord[i]
        z_local = z_local + propeller.com_shift_up[i] * chord[i]

    cos_t = np.cos(twist[i])
    sin_t = np.sin(twist[i])
    x_rot = x_local * cos_t + z_local * sin_t
    z_rot = -x_local * sin_t + z_local * cos_t

    X[i, :] = x_rot
    Y[i, :] = r[i]
    Z[i, :] = z_rot

    if sigma_total.ndim == 2 and sigma_total.shape[1] == n_points:
        S[i, :] = sigma_total[i, :]
    else:
        S[i, :] = sigma_total[i]

r_fine = np.linspace(r.min(), r.max(), 50)
X_fine = np.zeros((len(r_fine), n_points))
Z_fine = np.zeros((len(r_fine), n_points))
S_fine = np.zeros((len(r_fine), n_points))

for j in range(n_points):
    X_fine[:, j] = np.interp(r_fine, r, X[:, j])
    Z_fine[:, j] = np.interp(r_fine, r, Z[:, j])
    S_fine[:, j] = np.interp(r_fine, r, S[:, j])

Y_fine = np.repeat(r_fine[:, np.newaxis], n_points, axis=1)
S_mpa = S_fine / 1e6

fig = plt.figure(figsize=(4.5, 3))
ax = fig.add_subplot(111, projection='3d')

norm = Normalize(vmin=np.nanmin(S_mpa), vmax=np.nanmax(S_mpa))
ax.plot_surface(
    X_fine,
    Y_fine,
    Z_fine,
    facecolors=cmap(norm(S_mpa)),
    rstride=1,
    cstride=1,
    linewidth=0,
    antialiased=True,
    shade=False,
    edgecolor='none',
    vmin=-10,
    vmax=20,
)

z_min = np.nanmin(Z_fine)
z_max = np.nanmax(Z_fine)
pad = 0.01 * (z_max - z_min)
ax.set_zlim(z_min - pad, z_max + pad)

ax.set_axis_off()
ax.set_box_aspect((np.ptp(X_fine), np.ptp(Y_fine), np.ptp(Z_fine)))

vmin = np.floor(S_mpa.min() / 10) * 10
vmax = np.ceil(S_mpa.max() / 10) * 10

mappable = mpl.cm.ScalarMappable(norm=norm, cmap=cmap)
mappable.set_array(S_mpa)
cbar = fig.colorbar(mappable, ax=ax, shrink=0.4, pad=-0.05, aspect=10)
cbar.set_label('Stress [MPa]')
cbar.set_ticks(ticks=[-10, -5, 0, 5, 10, 15, 20])
ax.view_init(elev=15, azim=-30)
plt.tight_layout(pad=0)
fig.subplots_adjust(left=0, right=1, bottom=0, top=1, wspace=0, hspace=0)
plt.gca().set_aspect('auto', adjustable='box')
plt.savefig('../Figures/fig1.pdf', bbox_inches='tight', dpi=200)
plt.savefig('../Figures/pca_opt.svg', bbox_inches='tight', dpi=200)
plt.show()

MemoryError: Unable to allocate 364. GiB for an array with shape (48806539975,) and data type int64

Error in callback <function _draw_all_if_interactive at 0x000001983FAE4680> (for post_execute), with arguments args (),kwargs {}:


MemoryError: Unable to allocate 364. GiB for an array with shape (48806539975,) and data type int64

MemoryError: Unable to allocate 364. GiB for an array with shape (48806539975,) and data type int64

<Figure size 450x300 with 2 Axes>